In [ ]:
import pandas as pd

df = pd.read_csv("/home/eschaabbas/Dokumente/CSV/Join2.csv")

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/home/eschaabbas/Dokumente/CSV/Join3.csv",
    sep=",",
    quotechar='"',
    engine="python",
    on_bad_lines="skip"
)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

In [22]:
import pandas as pd

# CSV laden
df = pd.read_csv("/home/eschaabbas/Dokumente/CSV/Join2.csv")

# Optional: Spaltennamen prüfen
print(df.columns.tolist())

# Nach Ticket und Zeit sortieren
df = df.sort_values(["ticket_id", "create_time"]).copy()

samples = []

for ticket_id, group in df.groupby("ticket_id"):
    group = group.sort_values("create_time")
    
    customer_articles = group[group["article_sender_type_id"] == 3]
    agent_articles = group[group["article_sender_type_id"] == 1]

    if customer_articles.empty or agent_articles.empty:
        continue

    problem_row = customer_articles.iloc[0]
    solution_row = agent_articles.iloc[-1]

    problem_text = str(problem_row.get("body_text", "")).strip()
    solution_text = str(solution_row.get("body_text", "")).strip()

    if len(problem_text) < 20 or len(solution_text) < 20:
        continue

    sample = {
        "ticket_id": ticket_id,
        "ticket_number": problem_row.get("ticket_number"),
        "title": problem_row.get("title"),
        "queue": problem_row.get("queue"),
        "state": problem_row.get("state"),
        "problem_text": problem_text,
        "solution_text": solution_text,
    }

    samples.append(sample)

samples_df = pd.DataFrame(samples)

print("Anzahl Samples:", len(samples_df))
samples_df.head()

['ticket_id', 'ticket_number', 'title', 'queue', 'state', 'priority', 'article_id', 'create_time', 'communication_channel_id', 'article_sender_type_id', 'article_created_by']
Anzahl Samples: 0


""


In [19]:
import pandas as pd
import re
from pathlib import Path

CSV_PATH = Path("/home/eschaabbas/Dokumente/CSV/Join3.csv")
OUTPUT_DIR = Path("/home/eschaabbas/Dokumente/OTRS/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_TEXT_LEN = 20

SENDER_AGENT = 1
SENDER_SYSTEM = 2
SENDER_CUSTOMER = 3


def load_csv(csv_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        csv_path,
        sep=",",
        quotechar='"',
        engine="python",
        on_bad_lines="skip"
    )


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    header_patterns = [
        r"^Received:.*$",
        r"^Return-Path:.*$",
        r"^Delivered-To:.*$",
        r"^From:.*$",
        r"^To:.*$",
        r"^Cc:.*$",
        r"^BCC:.*$",
        r"^Subject:.*$",
        r"^Date:.*$",
        r"^Sent:.*$",
        r"^Reply-To:.*$",
        r"^Message-ID:.*$",
        r"^In-Reply-To:.*$",
        r"^References:.*$",
        r"^MIME-Version:.*$",
        r"^Content-Type:.*$",
        r"^Content-Transfer-Encoding:.*$",
        r"^X-.*$",
    ]

    for pattern in header_patterns:
        text = re.sub(pattern, "", text, flags=re.MULTILINE)

    text = re.sub(r"(?im)^>.*$", "", text)
    text = re.sub(r"(?im)^am .* schrieb .*:$", "", text)
    text = re.sub(r"(?im)^on .* wrote:$", "", text)

    text = re.sub(r"(?is)this e-mail.*confidential.*", "", text)
    text = re.sub(r"(?is)diese e-mail.*vertraulich.*", "", text)

    text = re.sub(r"(?im)^mit freundlichen grüßen.*", "", text)
    text = re.sub(r"(?im)^freundliche grüße.*", "", text)
    text = re.sub(r"(?im)^best regards.*", "", text)
    text = re.sub(r"(?im)^kind regards.*", "", text)
    text = re.sub(r"(?im)^--\s*$", "", text)

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " [URL] ", text)
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", "[EMAIL]", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "[IP]", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# 1) CSV robust laden
df = load_csv(CSV_PATH)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "ticket_id",
    "ticket_number",
    "title",
    "queue",
    "state",
    "priority",
    "article_id",
    "create_time",
    "communication_channel_id",
    "article_sender_type_id",
    "article_created_by",
    "body_text",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Fehlende Spalten in CSV: {missing_cols}")

# 2) Typen normalisieren
df["create_time"] = pd.to_datetime(df["create_time"], errors="coerce")

for col in ["ticket_id", "ticket_number", "article_id", "communication_channel_id", "article_sender_type_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["title", "queue", "state", "priority", "article_created_by", "body_text"]:
    df[col] = df[col].fillna("").astype(str)

# 3) Text bereinigen
df["body_text_clean"] = df["body_text"].apply(clean_text)

# 4) Relevante Artikel filtern
df_filtered = df[df["article_sender_type_id"].isin([SENDER_CUSTOMER, SENDER_AGENT])].copy()
df_filtered = df_filtered[df_filtered["body_text_clean"].str.len() >= MIN_TEXT_LEN].copy()
df_filtered = df_filtered.sort_values(["ticket_id", "create_time", "article_id"]).copy()

print("Gefilterte Artikel:", df_filtered.shape)

# 5) Problem-/Lösungspaare bauen
samples = []

for ticket_id, group in df_filtered.groupby("ticket_id"):
    group = group.sort_values(["create_time", "article_id"])

    customer_articles = group[group["article_sender_type_id"] == SENDER_CUSTOMER]
    agent_articles = group[group["article_sender_type_id"] == SENDER_AGENT]

    if customer_articles.empty or agent_articles.empty:
        continue

    problem_row = customer_articles.iloc[0]
    solution_row = agent_articles.iloc[-1]

    problem_text = str(problem_row.get("body_text_clean", "")).strip()
    solution_text = str(solution_row.get("body_text_clean", "")).strip()

    if len(problem_text) < MIN_TEXT_LEN or len(solution_text) < MIN_TEXT_LEN:
        continue

    samples.append({
        "ticket_id": int(problem_row["ticket_id"]) if pd.notna(problem_row["ticket_id"]) else None,
        "ticket_number": str(problem_row.get("ticket_number", "")).strip(),
        "title": str(problem_row.get("title", "")).strip(),
        "queue": str(problem_row.get("queue", "")).strip(),
        "state": str(problem_row.get("state", "")).strip(),
        "priority": str(problem_row.get("priority", "")).strip(),
        "problem_article_id": int(problem_row["article_id"]) if pd.notna(problem_row["article_id"]) else None,
        "solution_article_id": int(solution_row["article_id"]) if pd.notna(solution_row["article_id"]) else None,
        "problem_create_time": problem_row.get("create_time"),
        "solution_create_time": solution_row.get("create_time"),
        "problem_text": problem_text,
        "solution_text": solution_text,
    })

samples_df = pd.DataFrame(samples)

print("Anzahl Samples vor Qualitätsfilter:", len(samples_df))
print(samples_df.head())

# 6) Grober Qualitätsfilter
if not samples_df.empty:
    low_quality_pattern = r"^\s*(ok|erledigt|gelöst|danke|test)\s*$"

    samples_df = samples_df[samples_df["title"].str.len() >= 3]
    samples_df = samples_df[samples_df["problem_text"].str.len() >= 20]
    samples_df = samples_df[samples_df["solution_text"].str.len() >= 20]
    samples_df = samples_df[~samples_df["problem_text"].str.lower().str.match(low_quality_pattern, na=False)]
    samples_df = samples_df[~samples_df["solution_text"].str.lower().str.match(low_quality_pattern, na=False)]

print("Anzahl Samples nach Qualitätsfilter:", len(samples_df))
print(samples_df.head())

# 7) SFT-Dataset bauen
sft_rows = []

for _, row in samples_df.iterrows():
    sft_rows.append({
        "ticket_id": row["ticket_id"],
        "ticket_number": row["ticket_number"],
        "instruction": "Analysiere das IT-Support-Problem und liefere eine strukturierte Lösung.",
        "input": (
            f"Titel: {row['title']}\n"
            f"Queue: {row['queue']}\n"
            f"Status: {row['state']}\n"
            f"Problem:\n{row['problem_text']}"
        ),
        "output": f"Lösung:\n{row['solution_text']}",
    })

sft_df = pd.DataFrame(sft_rows)

print("SFT-Datensätze:", len(sft_df))
print(sft_df.head())

# 8) Export
raw_output_csv = OUTPUT_DIR / "otrs_ticket_pairs_raw.csv"
clean_output_csv = OUTPUT_DIR / "otrs_ticket_pairs_clean.csv"
sft_output_csv = OUTPUT_DIR / "otrs_sft_dataset.csv"
sft_output_jsonl = OUTPUT_DIR / "otrs_sft_dataset.jsonl"

df.to_csv(raw_output_csv, index=False)
samples_df.to_csv(clean_output_csv, index=False)
sft_df.to_csv(sft_output_csv, index=False)
sft_df.to_json(sft_output_jsonl, orient="records", lines=True, force_ascii=False)

print("\nExport abgeschlossen:")
print("RAW CSV:      ", raw_output_csv)
print("CLEAN CSV:    ", clean_output_csv)
print("SFT CSV:      ", sft_output_csv)
print("SFT JSONL:    ", sft_output_jsonl)

print("\n--- Statistik ---")
print("Rohdaten:", len(df))
print("Gefilterte Artikel:", len(df_filtered))
print("Sample-Paare:", len(samples_df))
print("SFT-Datensätze:", len(sft_df))

if not samples_df.empty:
    print("\nTop Queues:")
    print(samples_df["queue"].value_counts().head(10))

    print("\nTop Status:")
    print(samples_df["state"].value_counts().head(10))

Loaded: (3336, 12)
Columns: ['ticket_id', 'ticket_number', 'title', 'queue', 'state', 'priority', 'article_id', 'create_time', 'communication_channel_id', 'article_sender_type_id', 'article_created_by', 'body_text']
Gefilterte Artikel: (0, 13)
Anzahl Samples vor Qualitätsfilter: 0
Empty DataFrame
Columns: []
Index: []
Anzahl Samples nach Qualitätsfilter: 0
Empty DataFrame
Columns: []
Index: []
SFT-Datensätze: 0
Empty DataFrame
Columns: []
Index: []

Export abgeschlossen:
RAW CSV:       /home/eschaabbas/Dokumente/OTRS/otrs_ticket_pairs_raw.csv
CLEAN CSV:     /home/eschaabbas/Dokumente/OTRS/otrs_ticket_pairs_clean.csv
SFT CSV:       /home/eschaabbas/Dokumente/OTRS/otrs_sft_dataset.csv
SFT JSONL:     /home/eschaabbas/Dokumente/OTRS/otrs_sft_dataset.jsonl

--- Statistik ---
Rohdaten: 3336
Gefilterte Artikel: 0
Sample-Paare: 0
SFT-Datensätze: 0


/tmp/ipykernel_159191/1644449410.py:106: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["create_time"] = pd.to_datetime(df["create_time"], errors="coerce")
